# Component 2: final model training and export (Colab)

Trains the deployment model approved in `docs/PHASE7_RESULTS.md`: pv8-v1 features (physics8) and the 8-16-5 tanh MLP (229 parameters), with the same hyperparameters as the Phase 7 comparison.

- **Data:** `pv_fault_dataset.csv` only. Its SHA-256 is checked before training.
- **TRAINING window only.** V_ref, the scaler and the MLP weights are fitted on TRAINING rows. **The OPERATION window is not used** for training, seed choice, thresholds or evaluation. It stays reserved for the final test.
- **Seed:** 0, fixed before training by rule (`docs/TRAINING_EXPORT.md`). No seed is chosen by score.

Output: `model_contract.json`, `feature_config.json`, `training_metadata.json`, `manifest.json`. These are **locally trained weights**. They are not an FL update, not an INT8 build, and not deployed or field-validated.

All code comes from `component2-tinyml/src/tinyml/` (`train.py`, `export.py`) and `scripts/05_train_final.py`. This notebook only calls it.

Run the cells in order with **Shift + Enter**. Step 3 asks you to upload `pv_fault_dataset.csv`.

## Step 1: Get the latest repository code and install dependencies

In [ ]:
# STEP 1: Get the latest component2-tinyml code and install dependencies
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True, check=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

## Step 2: Import the repository modules

In [ ]:
# STEP 2: Import the repository modules
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import DATASET_SHA256, FINAL_SEED, FINAL_SEED_POLICY, check_dataset_sha256
from tinyml.export import N_PARAMS, PARAM_LAYOUT, verify_artifacts

spec = importlib.util.spec_from_file_location("train_final", f"{COMPONENT_DIR}/scripts/05_train_final.py")
train_final_script = importlib.util.module_from_spec(spec); spec.loader.exec_module(train_final_script)
print("Layout:", PARAM_LAYOUT, "=", N_PARAMS, "parameters")
print("Seed  :", FINAL_SEED, "|", FINAL_SEED_POLICY)

## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. The file stays only in this Colab session.

In [ ]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [ ]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

## Step 5: Train and export (TRAINING window only, a few seconds)
Fits the features and the MLP on the TRAINING rows, writes the four files to `/content/final_model/`, and verifies them (hashes, label/feature order, 229-parameter layout, reference probes).

In [ ]:
# STEP 5: Final training and export (no OPERATION data)
SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_DIR = "/content/final_model"
result, contract, metadata, paths = train_final_script.run(
    DATA_PATH, SPLITS_PATH, OUT_DIR, seed=FINAL_SEED, code_commit=COMMIT)
for p in paths:
    print("wrote", p)

## Step 6: Training summary
The training-fit accuracy is measured on the training rows themselves. It only shows that the fit worked; **it is not a performance result**. Performance estimates come from Phase 7 cross-validation; the final test is a separate, later step.

In [ ]:
# STEP 6: Training summary and artifact verification
f = result["facts"]
print(f"Rows {f['rows']} TRAINING ({f['runs']} runs, sites {f['sites']})")
print(f"Used {f['ok_rows_used']} | skipped {f['skipped_rows']} {f['skip_reasons']}")
print("Class counts:", f["class_counts"])
print(f"Seed {f['seed']} | iterations {f['n_iter']}/{f['max_iter']} | converged {f['converged']} | loss {f['final_loss']:.5f}")
print(f"Training-fit accuracy (not a performance result): {f['training_fit_accuracy']:.4f}")
print("Warnings:", f["warnings"] or "none")
print("V_ref:", contract["feature_config"]["vref"])
print("Weights SHA-256:", contract["weights_sha256"])
print("Environment:", contract["environment"], "| commit", contract["trained_on"]["git_commit"])
verify_artifacts(OUT_DIR)
print("Artifacts verified.")

## Step 7: Download the files
Downloads `final_model.zip` with the four files. **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [ ]:
# STEP 7: Zip and download
import shutil
zip_path = shutil.make_archive("/content/final_model", "zip", OUT_DIR)
files.download(zip_path)
print("Downloaded", zip_path)